# 03 -- Searching Memories

## What We'll Cover
1. Basic semantic search
2. Hybrid search (vector + keyword)
3. Threshold tuning for precision/recall
4. Pagination and result limits
5. Search with container scope
6. Using `profile()` as a combined context+search call


## 3.1 Two Search APIs

Supermemory offers two search interfaces:
- **`client.search.execute(q, ...)`** -- General search across everything
- **`client.search.documents(q, container_tags, ...)`** -- Scoped search within containers

Both support hybrid search (vector + keyword) with sub-300ms latency.


In [ ]:
# 3.2 Basic semantic search
from supermemory import Supermemory
client = Supermemory()

# Search across all accessible documents
results = client.search.execute(
    q="machine learning and AI concepts"
)

print(f"Found {len(results.results)} results:")
for i, r in enumerate(results.results[:5], 1):
    memory_text = r.get("memory", str(r))
    score = r.get("score", "N/A")
    print(f"  {i}. [{score}] {memory_text[:120]}...")


In [ ]:
# 3.3 Scoped search with container tags
# Search only within specific containers
results = client.search.documents(
    q="design systems",
    container_tags=["user_sarah"]
)

print(f"Found {len(results.results)} results in user_sarah:")
for r in results.results[:5]:
    print(f"  - {r.get('memory', str(r))[:100]}...")


In [ ]:
# 3.4 Threshold tuning -- balance precision vs recall
# threshold: 0.0-1.0 (higher = more precise, lower = more inclusive)

print("=== High Precision (threshold=0.7) ===")
results_precise = client.search.documents(
    q="distributed systems",
    document_threshold=0.7,
    limit=5
)
print(f"Results: {len(results_precise.results)}")

print("=== High Recall (threshold=0.3) ===")
results_broad = client.search.documents(
    q="distributed systems",
    document_threshold=0.3,
    limit=10
)
print(f"Results: {len(results_broad.results)}")

print("Tip: Use 0.7+ for exact matches, 0.3-0.5 for exploratory searches")


In [ ]:
# 3.5 Pagination with offset and limit
page_size = 3
page = 0

while True:
    results = client.search.documents(
        q="engineering",
        limit=page_size,
        offset=page * page_size
    )
    
    if not results.results:
        break

    print(f"--- Page {page + 1} ---")
    for r in results.results:
        print(f"  {r.get('memory', str(r))[:80]}...")
    page += 1

print(f"\nTotal pages: {page}")


In [ ]:
# 3.6 Speed optimization: onlyMatchingChunks
# When you only need the exact match snippets, skip full context
results = client.search.documents(
    q="Python programming",
    limit=5,
    only_matching_chunks=True  # Faster, returns only matching chunks
)
print(f"Fast search results: {len(results.results)}")
for r in results.results[:3]:
    print(f"  - {r.get('memory', str(r))[:100]}...")


In [ ]:
# 3.7 profile() is search + profile combined (recommended!)
# ONE call gets you: static facts + dynamic context + relevant memories
profile = client.profile(
    container_tag="user_sarah",
    q="design tools and preferences"
)

print("=== Static Facts (always available) ===")
for fact in profile.profile.static:
    print(f"  - {fact}")

print("\n=== Dynamic Facts (recent context) ===")
for fact in profile.profile.dynamic:
    print(f"  - {fact}")

print("\n=== Relevant Memories (from search) ===")
for r in profile.search_results.results[:3]:
    print(f"  - {r.get('memory', str(r))[:100]}...")

print("\nThis is the recommended pattern for chatbots and agents!")


## 3.8 Search Best Practices

- **Use `profile()` for chatbots** -- combines context + search in one call
- **Set `document_threshold`** -- 0.7+ for precise, 0.3-0.5 for exploratory
- **Keep limits small** -- `limit=5` is usually enough; paginate for more
- **Use `only_matching_chunks=True`** for speed when you don't need full context
- **Scope with container tags** -- searches without tags are slower and less relevant

**Next:** Notebook 04 -- User Profiles
